In [1]:
%load_ext autoreload
%autoreload 2
import sys; sys.path.append("..")
import numpy as np, pandas as pd, joblib
from src.data import load_train_val
from src.features import add_features, FEATURES, NUMERIC

pd.set_option("display.width", 200)
tr, val = load_train_val("../data/raw/fraudTrain.csv")
model = joblib.load("../models/xgb_v1.joblib")
val["score"] = model.predict_proba(val[FEATURES])[:, 1]

In [2]:
print("Transactions in both train and val:", tr["trans_num"].isin(val["trans_num"]).sum())

raw = pd.read_csv("../data/raw/fraudTrain.csv", index_col=0,
                  parse_dates=["trans_date_trans_time"])
cut = pd.Timestamp("2019-07-01")
full  = add_features(raw).set_index("trans_num")
early = add_features(raw[raw["trans_date_trans_time"] < cut]).set_index("trans_num")

same = np.isclose(full.loc[early.index, NUMERIC], early[NUMERIC], equal_nan=True)
print("Unchanged after deleting the future (all should be True):")
print(pd.Series(same.all(axis=0), index=NUMERIC))

# Prove the test can catch leakage: the leaky feature must FAIL
full_leaky  = full.groupby("cc_num")["amt"].transform("mean")
early_leaky = early.groupby("cc_num")["amt"].transform("mean")
print("Leaky feature passes? (should be False):",
      np.isclose(full_leaky.loc[early.index], early_leaky).all())

Transactions in both train and val: 0
Unchanged after deleting the future (all should be True):
amt                 True
hour                True
hours_since_prev    True
amt_vs_card_avg     True
txn_count_1h        True
amt_sum_1h          True
txn_count_24h       True
amt_sum_24h         True
dtype: bool
Leaky feature passes? (should be False): False


In [3]:
COSTS = {
    "otp_friction": 2.0,     # genuine customer asked for OTP (assumption)
    "block_friction": 30.0,  # genuine customer's card blocked (assumption)
    "otp_stop_rate": 0.8,    # share of frauds stopped by OTP (assumption)
}
y, amt, score = val["is_fraud"].to_numpy(), val["amt"].to_numpy(), val["score"].to_numpy()

def policy_cost(t_otp, t_block, c=COSTS):
    block = score >= t_block
    otp = (score >= t_otp) & ~block
    approve = ~(block | otp)
    fraud, legit = y == 1, y == 0
    fraud_loss = amt[fraud & approve].sum() + (1 - c["otp_stop_rate"]) * amt[fraud & otp].sum()
    friction = c["otp_friction"] * (legit & otp).sum() + c["block_friction"] * (legit & block).sum()
    return fraud_loss, friction

grid = [0.005, 0.01, 0.02, 0.05, 0.1, 0.2, 0.3, 0.5, 0.7, 0.9, 0.95, 0.99]
rows = []
for t_otp in grid:
    for t_block in grid + [1.01]:          # 1.01 means "never block"
        if t_block < t_otp:
            continue
        loss, friction = policy_cost(t_otp, t_block)
        rows.append({"t_otp": t_otp, "t_block": t_block, "fraud_loss": loss,
                     "friction": friction, "total": loss + friction})
policies = pd.DataFrame(rows).sort_values("total")

no_model = amt[y == 1].sum()
best = policies.iloc[0]
print(f"No model (approve everything): ${no_model:,.0f}")
print(policies.head(5).round(3).to_string(index=False))
print(f"Best policy cuts fraud-related cost by {1 - best['total'] / no_model:.1%}")

No model (approve everything): $1,220,266
 t_otp  t_block  fraud_loss  friction     total
 0.005      0.2   10278.612   13258.0 23536.612
 0.005      0.3   13019.672   10822.0 23841.672
 0.010      0.2   12777.636   11206.0 23983.636
 0.010      0.3   15518.696    8770.0 24288.696
 0.020      0.2   16436.892    9920.0 26356.892
Best policy cuts fraud-related cost by 98.1%


In [4]:
t_otp, t_block = best["t_otp"], best["t_block"]
val["action"] = np.select([val["score"] >= t_block, val["score"] >= t_otp],
                          ["block", "otp"], default="approve")
val["flagged"] = val["action"] != "approve"
print(pd.crosstab(val["is_fraud"].map({0: "legit", 1: "fraud"}), val["action"]))

action    approve  block   otp
is_fraud                      
fraud          21   2158   107
legit      366536    259  2744


In [5]:
both = pd.concat([tr, val[tr.columns]]).sort_values(["cc_num", "trans_date_trans_time"])
val["prev_is_fraud"] = both.groupby("cc_num")["is_fraud"].shift(fill_value=0).loc[val.index]

frauds = val[val["is_fraud"] == 1]
print(frauds.groupby("prev_is_fraud")["flagged"].agg(recall="mean", n="count")
      .rename(index={0: "first fraud on card", 1: "fraud after a fraud"}))

                       recall     n
prev_is_fraud                      
first fraud on card  0.970213   235
fraud after a fraud  0.993174  2051


In [6]:
cols = ["amt", "amt_vs_card_avg", "amt_sum_24h", "hours_since_prev", "hour"]
groups = {
    "caught fraud": val[(val["is_fraud"] == 1) & val["flagged"]],
    "missed fraud": val[(val["is_fraud"] == 1) & ~val["flagged"]],
    "false alarm":  val[(val["is_fraud"] == 0) & val["flagged"]],
    "normal legit": val[(val["is_fraud"] == 0) & ~val["flagged"]],
}
profile = pd.DataFrame({name: g[cols].median() for name, g in groups.items()}).T
profile["n"] = [len(g) for g in groups.values()]
print(profile.round(2).to_string())

                 amt  amt_vs_card_avg  amt_sum_24h  hours_since_prev  hour       n
caught fraud  443.02             5.66      1654.23              1.38  22.0    2265
missed fraud   19.59             0.34       277.95              9.61  18.0      21
false alarm   102.33             1.24       264.11              7.82  15.0    3003
normal legit   47.50             0.67       147.48              5.27  14.0  366536


In [7]:
max_train_fraud = tr.loc[tr["is_fraud"] == 1, "amt"].max()
print(f"Largest fraud amount in training: ${max_train_fraud:,.2f}")

caught = groups["caught fraud"]
probe = caught.copy()
probe["amt_vs_card_avg"] = probe["amt_vs_card_avg"] * (5000 / probe["amt"])
probe["amt"] = 5000.0
probe_scores = model.predict_proba(probe[FEATURES])[:, 1]

print(f"Caught frauds at real amounts: median score {caught['score'].median():.3f}")
print(f"Same frauds at $5,000:         median score {np.median(probe_scores):.3f}")
print(f"Would now be approved: {(probe_scores < t_otp).mean():.1%}")

Largest fraud amount in training: $1,371.81
Caught frauds at real amounts: median score 0.996
Same frauds at $5,000:         median score 0.990
Would now be approved: 0.3%


In [8]:
import json

def best_policy(costs, grid):
    rows = []
    for t_otp in grid:
        for t_block in grid + [1.01]:
            if t_block < t_otp:
                continue
            loss, friction = policy_cost(t_otp, t_block, costs)
            rows.append({"t_otp": t_otp, "t_block": t_block, "fraud_loss": loss,
                         "friction": friction, "total": loss + friction})
    return pd.DataFrame(rows).sort_values("total").iloc[0]

fine_grid = [0.0005, 0.001, 0.002, 0.003, 0.005, 0.0075, 0.01, 0.02, 0.05,
             0.1, 0.15, 0.2, 0.25, 0.3, 0.5, 0.7, 0.9]
scenarios = {
    "base":             COSTS,
    "expensive blocks": {**COSTS, "block_friction": 100.0},
    "annoying OTP":     {**COSTS, "otp_friction": 5.0},
    "weak OTP":         {**COSTS, "otp_stop_rate": 0.5},
}
summary = pd.DataFrame({name: best_policy(c, fine_grid) for name, c in scenarios.items()}).T
summary["cost_cut"] = 1 - summary["total"] / no_model
print(summary.round(4).to_string())

chosen = summary.loc["base"]
with open("../models/policy.json", "w") as f:
    json.dump({"t_otp": float(chosen["t_otp"]), "t_block": float(chosen["t_block"]),
               "costs": COSTS}, f, indent=2)

                   t_otp  t_block  fraud_loss  friction      total  cost_cut
base              0.0075     0.25   12387.032   10604.0  22991.032    0.9812
expensive blocks  0.0075     0.50   20191.450   14006.0  34197.450    0.9720
annoying OTP      0.0100     0.25   13954.712   15160.0  29114.712    0.9761
weak OTP          0.0075     0.10   15485.080   18108.0  33593.080    0.9725
